**Program 2: Classical and Distributed Word Representation Techniques**

**Aim**

To construct Bag-of-Words and TF-IDF representations and compare them with Word2Vec
or GloVe embeddings for similarity and downstream classification.

**Problem Statement**

News Topic Similarity and Routing: A digital news desk wants related stories grouped and
incoming articles routed to World, Sports, Business, or Science/Technology. Students
represent AG News descriptions with BoW, TF-IDF, and Word2Vec, compare document
similarity, and train the same classifier on each representation.

**Software Requirements**

• Python 3.x

• Scikit-learn

• Gensim

• NumPy and Pandas

• Matplotlib or Seaborn

In [1]:
!pip install datasets gensim scikit-learn pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 32.8 MB/s  0:00:00


**Step 1: Prepare the corpus**

• Apply the preprocessing pipeline from Program 1.

• Split data into training and testing sets using stratification where applicable.

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

ds = load_dataset("fancyzhx/ag_news")

df_train = pd.DataFrame(ds['train']).groupby('label', group_keys=False).apply(lambda x: x.sample(2000, random_state=42)).reset_index(drop=True)
df_test = pd.DataFrame(ds['test']).groupby('label', group_keys=False).apply(lambda x: x.sample(500, random_state=42)).reset_index(drop=True)

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_train['clean_text'] = df_train['text'].apply(preprocess)
df_test['clean_text'] = df_test['text'].apply(preprocess)

X_train = df_train['clean_text']
y_train = df_train['label']
X_test = df_test['clean_text']
y_test = df_test['label']

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

/tmp/ipykernel_509/4154590967.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train = pd.DataFrame(ds['train']).groupby('label', group_keys=False).apply(lambda x: x.sample(2000, random_state=42)).reset_index(drop=True)
/tmp/ipykernel_509/4154590967.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test = pd.DataFrame(ds['test']).groupby('label', group_keys=False).apply(lambda x: x.sample(500, r

**Step 2: Create Bag-of-Words features**

• Fit CountVectorizer on training data only.

• Inspect vocabulary size, sparse matrix shape, and frequent terms.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_bow = CountVectorizer(max_features=5000)
X_train_bow = vectorizer_bow.fit_transform(X_train)
X_test_bow = vectorizer_bow.transform(X_test)

print(len(vectorizer_bow.vocabulary_))
print(X_train_bow.shape)

5000
(8000, 5000)


**Step 3: Create TF-IDF features**

• Fit TfidfVectorizer using unigram and optional bigram features.

• Inspect high-weight terms for representative documents.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

print(X_train_tfidf.shape)

(8000, 5000)


**Step 4: Create distributed embeddings**

• Train Word2Vec using CBOW or Skip-gram, or load suitable pretrained GloVe vectors.

• Create document vectors by averaging token embeddings.

In [5]:
from gensim.models import Word2Vec

sentences_train = [text.split() for text in X_train]
sentences_test = [text.split() for text in X_test]

w2v_model = Word2Vec(sentences=sentences_train, vector_size=100, window=5, min_count=2, workers=4)

def get_doc_vector(tokens, model):
    valid_tokens = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_tokens:
        return np.zeros(model.vector_size)
    return np.mean(valid_tokens, axis=0)

X_train_w2v = np.array([get_doc_vector(tokens, w2v_model) for tokens in sentences_train])
X_test_w2v = np.array([get_doc_vector(tokens, w2v_model) for tokens in sentences_test])

**Step 5: Analyse similarity**

• Compute cosine similarity for selected word and document pairs.

• Identify nearest neighbours and discuss semantic quality.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

print(w2v_model.wv.most_similar('technology'))
print(w2v_model.wv.most_similar('sports'))

doc1 = X_train_w2v[0].reshape(1, -1)
doc2 = X_train_w2v[1].reshape(1, -1)
print(cosine_similarity(doc1, doc2))

[('further', 0.9981304407119751), ('results', 0.9981248378753662), ('trading', 0.9980233907699585), ('continued', 0.9980100989341736), ('body', 0.9977698922157288), ('such', 0.997658908367157), ('concerns', 0.9974993467330933), ('demand', 0.997450053691864), ('energy', 0.9974198341369629), ('financial', 0.9973099827766418)]
[('mln', 0.9973611235618591), ('ukraine', 0.9972049593925476), ('walmart', 0.9970571398735046), ('japans', 0.9968289732933044), ('mets', 0.9966039061546326), ('islamabad', 0.9964420795440674), ('warns', 0.9962279796600342), ('diego', 0.9959774613380432), ('defending', 0.9958173632621765), ('network', 0.9957635998725891)]
[[0.9931229]]


**Step 6: Compare representations**

• Train the same baseline classifier on BoW, TF-IDF, and averaged embeddings.

• Compare accuracy or F1-score, dimensionality, sparsity, training time, and out-of-
vocabulary behaviour.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import time

def train_and_evaluate(X_tr, y_tr, X_te, y_te, name):
    start = time.time()
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)
    acc = accuracy_score(y_te, preds)
    f1 = f1_score(y_te, preds, average='macro')
    end = time.time()
    print(f"{name} - Accuracy: {acc:.4f}, F1: {f1:.4f}, Time: {end-start:.4f}s")

train_and_evaluate(X_train_bow, y_train, X_test_bow, y_test, "Bag-of-Words")
train_and_evaluate(X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF")
train_and_evaluate(X_train_w2v, y_train, X_test_w2v, y_test, "Word2Vec")

Bag-of-Words - Accuracy: 0.8620, F1: 0.8622, Time: 5.5959s
TF-IDF - Accuracy: 0.8835, F1: 0.8834, Time: 1.8970s
Word2Vec - Accuracy: 0.5630, F1: 0.5625, Time: 5.2411s


**Expected Result & Observations to Record
Dataset split:**

* Stratified 8,000 train (2,000 per class) and 2,000 test (500 per class).

* Preprocessing: Lowercased and removed special characters.

* Evaluation metrics: The final output lists Accuracy, F1-Score, and training execution time for Bag-of-Words, TF-IDF, and Word2Vec models to allow direct comparison of sparse vs. distributed representations.